# Producer application

Santiago Elí Jiménez Aguilar  
Luis Eduardo Gonzalez Gloria

Teams will generate the same informationfrom the Final Project Part I and write it into a Kafka Topic to simulate a streaming the data
source.<br>
Evaluation criteria: clarity of the producer description and the link to the Pull Request containing the changes of the Producer code.

## Instructions
- **Topic**. Create Kafka topic.
- **Data Stream**. Create data stream.
- **Load Dataset**. Load generated dataset.
- **Send Events**. Send events to Kafka.

### Create SparkSession

In [1]:
from spark_utils import SparkUtils
from pathlib import Path
import shutil
import pyspark.sql.functions as F

mongodb_connector = "org.mongodb.spark:mongo-spark-connector_2.13:10.5.0"
kafka_connector = "org.apache.spark:spark-sql-kafka-0-10_2.13:4.0.0"
su = SparkUtils("FinalProject_kafka_mongodb",
                "spark://spark-master:7077",
                spark_packages=f"{kafka_connector},{mongodb_connector}")
su.spark

:: loading settings :: url = jar:file:/opt/spark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /root/.ivy2.5.2/cache
The jars for the packages stored in: /root/.ivy2.5.2/jars
org.apache.spark#spark-sql-kafka-0-10_2.13 added as a dependency
org.mongodb.spark#mongo-spark-connector_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-b790b75a-8ca8-4a95-a4ec-800abc707861;1.0
	confs: [default]
	found org.apache.spark#spark-sql-kafka-0-10_2.13;4.0.0 in central
	found org.apache.spark#spark-token-provider-kafka-0-10_2.13;4.0.0 in central
	found org.apache.kafka#kafka-clients;3.9.0 in central
	found org.lz4#lz4-java;1.8.0 in central
	found org.xerial.snappy#snappy-java;1.1.10.7 in central
	found org.slf4j#slf4j-api;2.0.16 in central
	found org.apache.hadoop#hadoop-client-runtime;3.4.1 in central
	found org.apache.hadoop#hadoop-client-api;3.4.1 in central
	found com.google.code.findbugs#jsr305;3.0.0 in central


### Create a data stream from a Kafka topic

#### Create `server-logs` topic

```
    docker exec -it <Kafka container ID> \
     /opt/kafka/bin/kafka-topics.sh \
      --create --zookeeper zookeeper:2181 \
      --replication-factor 1 --partitions 1 \
      --topic server-logs
```

```
    docker exec -it 0cf75da0ca5041b295ed8cee66adb76f233374978df08462f1a714cf961fdf8a /opt/kafka/bin/kafka-topics.sh --create --zookeeper zookeeper:2181 --replication-factor 1 --partitions 1 --topic server-logs
```

In [2]:
!pip install kafka-python

### Load generated data

In [3]:
import csv
import json
import time
import random
from kafka import KafkaProducer

BROKER = "kafka:9093"
TOPIC  = "watch-events"

DATA_PATH = "/opt/spark/work-dir/data/watch_history/watch_history.csv"

watch_rows = []
with open(DATA_PATH, newline="", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for row in reader:
        watch_rows.append(row)

print(f"Loaded {len(watch_rows)} watch events from: {DATA_PATH}")
print("\nSample record:")
print(json.dumps(watch_rows[0], indent=2))

Loaded 400 watch events from: /opt/spark/work-dir/data/watch_history/watch_history.csv

Sample record:
{
  "event_id": "f485d2da-4b43-4c86-82af-c889b63baeca",
  "user_id": "U0000018",
  "content_id": "C0000050",
  "watch_timestamp": "2026-07-18 11:42:47",
  "device": "Wii",
  "watch_time_minutes": "138",
  "completion_percentage": "24.25",
  "is_finished": "False",
  "region": "Costa Rica"
}


### Send events to kafka

Each row of `watch_history.csv` is serialized as JSON and published to the `watch-events` topic.  
Delay of 0.5 seconds between records to simulate a live data stream.

In [ ]:
producer = KafkaProducer(
    bootstrap_servers=BROKER,
    value_serializer=lambda v: json.dumps(v).encode("utf-8"),
)

print(f"Connected to broker : {BROKER}")
print(f"Topic               : {TOPIC}")
print(f"Records to send     : {len(watch_rows)}")
print(f"Delay of records: 0.5 seconds")
print("-" * 55)

for i, row in enumerate(watch_rows, 1):
    producer.send(TOPIC, value=row)
    delay = 0.5
    print(
        f"[{i:>3}] Sent: event_id={row['event_id'][:8]}... "
        f"user={row['user_id']}  content={row['content_id']}  "
        f"completion={row['completion_percentage']}%"
    )
    time.sleep(delay)

producer.flush()
producer.close()
print(f"\nDone. Total records sent: {len(watch_rows)}")

Connected to broker : kafka:9093
Topic               : watch-events
Records to send     : 400
Delay of records: 0.5 seconds
-------------------------------------------------------
[  1] Sent: event_id=f485d2da... user=U0000018  content=C0000050  completion=24.25%
[  2] Sent: event_id=93534524... user=U0000054  content=C0000084  completion=74.48%
[  3] Sent: event_id=8851aae8... user=U0000057  content=C0000057  completion=69.34%
[  4] Sent: event_id=f2bb4550... user=U0000046  content=C0000008  completion=90.27%
[  5] Sent: event_id=848944e4... user=U0000082  content=C0000077  completion=66.61%
[  6] Sent: event_id=6b162683... user=U0000009  content=C0000098  completion=78.91%
[  7] Sent: event_id=f394a3c0... user=U0000077  content=C0000001  completion=77.66%
[  8] Sent: event_id=d66c4804... user=U0000004  content=C0000017  completion=11.88%
[  9] Sent: event_id=81413e73... user=U0000034  content=C0000007  completion=33.05%
[ 10] Sent: event_id=9420bf39... user=U0000047  content=C0000066

In [ ]:
su.spark.stop()